# 🌤️ Proof of Concept: Data Ingestion Cuaca Open-Meteo API

**Proyek:** Sistem Otomatis Deteksi Anomali Data Cuaca Open-Meteo untuk Smart Farming Berbasis MLOps

| | |
|---|---|
| **Nama** | Muhammad Ghazy Humaidi |
| **NIM** | 245150200111071 |
| **Mata Kuliah** | MLOps - TIF-B 2026 |
| **Dosen** | Rizal Setya Perdana, S.Kom., M.Kom., Ph.D. |

---

## 📋 Deskripsi Notebook

Notebook ini merupakan **Proof of Concept (PoC)** yang membuktikan bahwa:
1. Data cuaca dapat diambil secara otomatis dari **Open-Meteo API**
2. Data bersifat **Tabular Time-Series** yang cocok untuk klasifikasi anomali
3. Rule-based anomaly detection dapat mengklasifikasikan data ke **3 kelas** (Normal, Hardware Fault, Extreme Weather)
4. Data dapat disimpan untuk proses training model selanjutnya

## 1. 📦 Setup & Dependencies

Install dan import library yang dibutuhkan.

In [ ]:
# Install dependencies (jika belum ada)
# !pip install requests pandas matplotlib scikit-learn

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# Konfigurasi display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

print(f"Notebook dijalankan pada: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. 🌐 Data Source: Open-Meteo API

Proyek ini menggunakan **[Open-Meteo API](https://open-meteo.com/)** sebagai sumber data tunggal.

**Kenapa Open-Meteo?**
- ✅ Gratis (Open-Access, tanpa API key)
- ✅ Data agrikultur lengkap (suhu, kelembaban, curah hujan, kelembaban tanah)
- ✅ Update setiap jam (cocok untuk micro-batch fetching)
- ✅ Mendukung data historis dan forecast

**Lokasi:** Malang, Jawa Timur (Lat: -7.95, Lon: 112.61)

## 3. 📥 Data Ingestion

Fungsi untuk mengambil data cuaca dari Open-Meteo API dengan variabel meteorologi yang relevan untuk Smart Agriculture.

In [ ]:
def fetch_weather_data(past_days=7, forecast_days=1):
    """
    Mengambil data cuaca dari Open-Meteo API untuk area Malang.
    
    Parameters:
        past_days (int): Jumlah hari ke belakang untuk data historis
        forecast_days (int): Jumlah hari ke depan untuk forecast
    
    Returns:
        pd.DataFrame: Data cuaca per jam
    """
    print("Memulai proses Ingestion Data dari Open-Meteo API...")
    
    # Konfigurasi endpoint dan parameter
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": -7.95,
        "longitude": 112.61,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "soil_moisture_0_to_7cm",
            "shortwave_radiation",
            "wind_speed_10m"
        ],
        "timezone": "Asia/Jakarta",
        "past_days": past_days,
        "forecast_days": forecast_days,
    }
    
    # Hit API
    response = requests.get(url, params=params, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        print(f"✅ Data berhasil di-fetch (Status: {response.status_code})")
        
        # Ekstrak JSON ke DataFrame
        df = pd.DataFrame({
            "timestamp": data["hourly"]["time"],
            "temperature_2m_C": data["hourly"]["temperature_2m"],
            "humidity_percent": data["hourly"]["relative_humidity_2m"],
            "precipitation_mm": data["hourly"]["precipitation"],
            "soil_moisture": data["hourly"]["soil_moisture_0_to_7cm"],
            "radiation_wm2": data["hourly"]["shortwave_radiation"],
            "wind_speed_kmh": data["hourly"]["wind_speed_10m"],
        })
        
        # Convert timestamp
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        
        # Metadata
        print(f"📍 Lokasi: Lat {data['latitude']}, Lon {data['longitude']}")
        print(f"🕐 Timezone: {data['timezone']}")
        print(f"📊 Total data points: {len(df)} baris (update per jam)")
        print(f"📅 Range: {df['timestamp'].min()} → {df['timestamp'].max()}")
        
        return df
    else:
        print(f"❌ Gagal mengambil data. Status code: {response.status_code}")
        return None

# Eksekusi: ambil data 7 hari terakhir + 1 hari forecast
df_cuaca = fetch_weather_data(past_days=7, forecast_days=1)

## 4. 🔍 Exploratory Data Analysis (EDA)

Eksplorasi data untuk memahami karakteristik dan distribusi variabel cuaca.

### 4.1 Snapshot Data

In [ ]:
if df_cuaca is not None:
    print("=" * 60)
    print("📋 5 BARIS PERTAMA (SNAPSHOT DATA)")
    print("=" * 60)
    display(df_cuaca.head())
    
    print("\n" + "=" * 60)
    print("📋 5 BARIS TERAKHIR")
    print("=" * 60)
    display(df_cuaca.tail())

### 4.2 Informasi Struktur Data

In [ ]:
if df_cuaca is not None:
    print("=" * 60)
    print("📊 INFO STRUKTUR DATA")
    print("=" * 60)
    print(f"Shape: {df_cuaca.shape}")
    print(f"Columns: {list(df_cuaca.columns)}")
    print()
    df_cuaca.info()
    
    print("\n" + "=" * 60)
    print("📊 MISSING VALUES")
    print("=" * 60)
    missing = df_cuaca.isnull().sum()
    print(missing)
    print(f"\nTotal missing values: {missing.sum()}")

### 4.3 Statistik Deskriptif

In [ ]:
if df_cuaca is not None:
    print("=" * 60)
    print("📊 STATISTIK DESKRIPTIF")
    print("=" * 60)
    display(df_cuaca.describe())

## 5. 📈 Visualisasi Data Cuaca

Visualisasi time-series untuk setiap variabel meteorologi.

In [ ]:
if df_cuaca is not None:
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle('Data Cuaca Malang - Open-Meteo API', fontsize=16, fontweight='bold')
    
    # Temperature
    axes[0, 0].plot(df_cuaca['timestamp'], df_cuaca['temperature_2m_C'], color='red', linewidth=0.8)
    axes[0, 0].set_title('🌡️ Suhu Udara (°C)')
    axes[0, 0].set_ylabel('°C')
    axes[0, 0].axhline(y=33, color='orange', linestyle='--', alpha=0.7, label='Threshold Anomali (33°C)')
    axes[0, 0].legend(fontsize=8)
    
    # Humidity
    axes[0, 1].plot(df_cuaca['timestamp'], df_cuaca['humidity_percent'], color='blue', linewidth=0.8)
    axes[0, 1].set_title('💧 Kelembaban Relatif (%)')
    axes[0, 1].set_ylabel('%')
    
    # Precipitation
    axes[1, 0].bar(df_cuaca['timestamp'], df_cuaca['precipitation_mm'], color='teal', width=0.03)
    axes[1, 0].set_title('🌧️ Curah Hujan (mm)')
    axes[1, 0].set_ylabel('mm')
    
    # Soil Moisture
    axes[1, 1].plot(df_cuaca['timestamp'], df_cuaca['soil_moisture'], color='brown', linewidth=0.8)
    axes[1, 1].set_title('🌱 Kelembaban Tanah (m³/m³)')
    axes[1, 1].set_ylabel('m³/m³')
    
    # Radiation
    axes[2, 0].fill_between(df_cuaca['timestamp'], df_cuaca['radiation_wm2'], color='gold', alpha=0.6)
    axes[2, 0].set_title('☀️ Radiasi Matahari (W/m²)')
    axes[2, 0].set_ylabel('W/m²')
    
    # Wind Speed
    axes[2, 1].plot(df_cuaca['timestamp'], df_cuaca['wind_speed_kmh'], color='gray', linewidth=0.8)
    axes[2, 1].set_title('💨 Kecepatan Angin (km/h)')
    axes[2, 1].set_ylabel('km/h')
    
    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=45, labelsize=8)
    
    plt.tight_layout()
    plt.show()

### 5.1 Heatmap Korelasi Antar Variabel

In [ ]:
if df_cuaca is not None:
    numeric_cols = ['temperature_2m_C', 'humidity_percent', 'precipitation_mm', 
                    'soil_moisture', 'radiation_wm2', 'wind_speed_kmh']
    corr_matrix = df_cuaca[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
    
    ax.set_xticks(range(len(numeric_cols)))
    ax.set_yticks(range(len(numeric_cols)))
    ax.set_xticklabels([c.replace('_', '\n') for c in numeric_cols], fontsize=8, rotation=45, ha='right')
    ax.set_yticklabels([c.replace('_', '\n') for c in numeric_cols], fontsize=8)
    
    for i in range(len(numeric_cols)):
        for j in range(len(numeric_cols)):
            ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
    
    plt.colorbar(im, ax=ax, label='Korelasi')
    plt.title('Heatmap Korelasi Variabel Cuaca', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. 🚨 Rule-Based Anomaly Detection (3 Class)

Simulasi deteksi anomali menggunakan **rule-based approach** sesuai desain di LK01:

| Class | Label | Kriteria |
|---|---|---|
| **0** | Normal State | Semua sensor dalam rentang operasional wajar |
| **1** | Hardware Fault Anomaly | Nilai stuck/konstan, NULL, atau di luar batas fisik |
| **2** | Extreme Environmental Anomaly | Suhu >33°C, kelembaban tanah sangat rendah, curah hujan ekstrem |

In [ ]:
def classify_anomaly(row):
    """
    Mengklasifikasikan setiap baris data ke dalam 3 kelas anomali.
    
    Class 0: Normal State
    Class 1: Hardware Fault Anomaly
    Class 2: Extreme Environmental Anomaly
    """
    # === CLASS 1: Hardware Fault Detection ===
    # Cek nilai NULL atau NaN
    if pd.isna(row['temperature_2m_C']) or pd.isna(row['humidity_percent']):
        return 1, 'Hardware Fault: Missing sensor reading'
    
    # Cek nilai di luar batas fisik yang mungkin (sensor rusak)
    if row['temperature_2m_C'] < -20 or row['temperature_2m_C'] > 55:
        return 1, 'Hardware Fault: Temperature out of physical bounds'
    
    if row['humidity_percent'] < 0 or row['humidity_percent'] > 100:
        return 1, 'Hardware Fault: Humidity out of bounds'
    
    if row['precipitation_mm'] < 0:
        return 1, 'Hardware Fault: Negative precipitation'
    
    # === CLASS 2: Extreme Environmental Anomaly ===
    # Suhu ekstrem tinggi (gelombang panas)
    if row['temperature_2m_C'] > 33:
        return 2, 'Extreme Weather: Heat wave (>33°C)'
    
    # Suhu ekstrem rendah (untuk wilayah tropis Malang)
    if row['temperature_2m_C'] < 15:
        return 2, 'Extreme Weather: Unusual cold (<15°C)'
    
    # Curah hujan sangat tinggi
    if row['precipitation_mm'] > 20:
        return 2, 'Extreme Weather: Heavy rainfall (>20mm/hr)'
    
    # Kelembaban tanah sangat rendah (kekeringan)
    if row['soil_moisture'] < 0.05:
        return 2, 'Extreme Weather: Severe drought (soil moisture <0.05)'
    
    # === CLASS 0: Normal State ===
    return 0, 'Normal'


if df_cuaca is not None:
    # Apply classification
    results = df_cuaca.apply(classify_anomaly, axis=1)
    df_cuaca['anomaly_class'] = results.apply(lambda x: x[0])
    df_cuaca['anomaly_reason'] = results.apply(lambda x: x[1])
    
    # Tampilkan hasil
    print("=" * 60)
    print("🚨 HASIL RULE-BASED ANOMALY DETECTION")
    print("=" * 60)
    
    class_labels = {
        0: 'Class 0: Normal State',
        1: 'Class 1: Hardware Fault',
        2: 'Class 2: Extreme Environmental'
    }
    
    class_counts = df_cuaca['anomaly_class'].value_counts().sort_index()
    for cls, count in class_counts.items():
        pct = count / len(df_cuaca) * 100
        print(f"  {class_labels[cls]}: {count} baris ({pct:.1f}%)")
    
    print(f"\nTotal data: {len(df_cuaca)} baris")
    
    # Tampilkan sample anomali (jika ada)
    anomalies = df_cuaca[df_cuaca['anomaly_class'] != 0]
    if not anomalies.empty:
        print(f"\n⚠️ Ditemukan {len(anomalies)} data anomali:")
        display(anomalies[['timestamp', 'temperature_2m_C', 'humidity_percent', 
                          'precipitation_mm', 'soil_moisture', 'anomaly_class', 
                          'anomaly_reason']].head(10))
    else:
        print("\n✅ Semua data dalam kondisi normal (Class 0).")

### 6.1 Visualisasi Distribusi Kelas Anomali

In [ ]:
if df_cuaca is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Distribusi kelas
    class_counts = df_cuaca['anomaly_class'].value_counts().sort_index()
    colors = ['#2ecc71', '#e74c3c', '#f39c12']
    labels = ['Class 0:\nNormal', 'Class 1:\nHardware Fault', 'Class 2:\nExtreme Weather']
    bars = axes[0].bar(range(len(class_counts)), class_counts.values, 
                       color=[colors[i] for i in class_counts.index])
    axes[0].set_xticks(range(len(class_counts)))
    axes[0].set_xticklabels([labels[i] for i in class_counts.index])
    axes[0].set_title('Distribusi Kelas Anomali', fontweight='bold')
    axes[0].set_ylabel('Jumlah Data Points')
    for bar, val in zip(bars, class_counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                    str(val), ha='center', fontweight='bold')
    
    # Plot 2: Time-series dengan anomali markers
    color_map = {0: '#2ecc71', 1: '#e74c3c', 2: '#f39c12'}
    for cls in df_cuaca['anomaly_class'].unique():
        mask = df_cuaca['anomaly_class'] == cls
        axes[1].scatter(df_cuaca.loc[mask, 'timestamp'], 
                       df_cuaca.loc[mask, 'temperature_2m_C'],
                       c=color_map[cls], label=labels[cls].replace('\n', ' '),
                       s=10, alpha=0.7)
    axes[1].set_title('Suhu dengan Label Anomali', fontweight='bold')
    axes[1].set_ylabel('Temperature (°C)')
    axes[1].legend(fontsize=8)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 7. 💾 Simpan Data ke CSV

Simpan data yang sudah di-fetch dan di-label ke folder `data/raw/` untuk digunakan pada proses training selanjutnya.

In [ ]:
if df_cuaca is not None:
    # Buat direktori jika belum ada
    os.makedirs('../data/raw', exist_ok=True)
    
    # Simpan ke CSV
    filename = f"../data/raw/weather_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_cuaca.to_csv(filename, index=False)
    print(f"✅ Data berhasil disimpan ke: {filename}")
    print(f"   Ukuran file: {os.path.getsize(filename) / 1024:.1f} KB")
    print(f"   Total baris: {len(df_cuaca)}")
    print(f"   Kolom: {list(df_cuaca.columns)}")

## 8. 📝 Kesimpulan & Next Steps

### ✅ Apa yang sudah dibuktikan:
1. **Data Ingestion** — Data cuaca berhasil di-fetch dari Open-Meteo API secara otomatis
2. **Modalitas Data** — Data bersifat Tabular Time-Series dengan atribut numerik kontinyu
3. **Anomaly Detection** — Rule-based classification berhasil membedakan 3 kelas anomali
4. **Data Persistence** — Data dapat disimpan ke CSV untuk proses selanjutnya

### 🔜 Next Steps (Pertemuan selanjutnya):
1. **Data Pipeline Automation** — Menggunakan Apache Airflow untuk scheduling ingestion per jam
2. **Data Versioning** — Implementasi DVC untuk tracking versi dataset
3. **Model Training** — Training model XGBoost/Random Forest untuk klasifikasi anomali
4. **Experiment Tracking** — MLflow untuk mencatat eksperimen dan model registry
5. **Model Serving** — Deploy model sebagai REST API menggunakan FastAPI
6. **Monitoring** — Evidently AI untuk deteksi data drift dan concept drift

---